# Manufacturing Efficiency Prediction Training Code

This notebook trains a model to predict the **Efficiency_Score** of a manufacturing process.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

print("Libraries imported.")

### Step 2: Load and Preprocess Data

In [ ]:
# Upload manufacturing_dataset.csv to Colab!
df = pd.read_csv('manufacturing_dataset.csv')

# Drop Timestamp
df = df.drop('Timestamp', axis=1)

# Define Target and Features
target = 'Efficiency_Score'
X = df.drop(target, axis=1)
y = df[target].values

# Identify Numerical and Categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Save Preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')
print("Data preprocessed and preprocessor.pkl saved.")

### Step 3: Define Pytorch Model

In [ ]:
input_dim = X_train_processed.shape[1]

class EfficiencyNet(nn.Module):
    def __init__(self, input_size):
        super(EfficiencyNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model = EfficiencyNet(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"Model initialized with input dim: {input_dim}")

### Step 4: Training

In [ ]:
X_train_t = torch.FloatTensor(X_train_processed)
y_train_t = torch.FloatTensor(y_train).view(-1, 1)

epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}')

print("Training complete.")

### Step 5: Export Model

In [ ]:
torch.save(model.state_dict(), 'efficiency_model.pth')
print("Model saved as efficiency_model.pth")